In [50]:
'''
tutorial:
https://www.kaggle.com/code/alexandervc/gene-ontology-python-tutorial
'''

'\ntutorial:\nhttps://www.kaggle.com/code/alexandervc/gene-ontology-python-tutorial\n'

In [51]:
from goatools import obo_parser

In [52]:
# obo file contains the go term database
go_obo = 'data/go-basic.obo'

# reads obo file to dictionary
go = obo_parser.GODag(go_obo)

Exception: COULD NOT READ(data/go-basic.obo)
download obo file first
 [http://geneontology.org/ontology/go-basic.obo]

In [ ]:
go_id = 'GO:0048527'
go_term = go[go_id]

In [ ]:
go_term

In [ ]:
'GO term name: {}'.format(go_term.name)

In [ ]:
rec = go[go_id]
rec

In [ ]:
# get parents of a go term
parents = rec.get_all_parents()
parents

In [ ]:
# get all childrn of a go term
children = rec.get_all_children()
children

In [ ]:
for term in parents.union(children):
    print(go[term])

In [ ]:
# section 4 go enrichment or depletion analysis
from goatools.go_enrichment import GOEnrichmentStudy

In [ ]:
# Sanbomics tutorial
'''
https://www.youtube.com/watch?v=ONiWugVEf2s&t=1s
'''

In [ ]:
#this part is not important, only a simple example of scanpy processing to get a list of genes
import numpy as np
import scanpy as sc
import pandas as pd
from matplotlib.pyplot import rc_context

my_data_path = 'path/to/data/outs/filtered_feature_bc_matrix'

def pp(path):
    adata = sc.read_10x_mtx(path)
    sc.pp.filter_cells(adata, min_genes=200) #get rid of cells with fewer than 200 genes
    sc.pp.filter_genes(adata, min_cells=3) #get rid of genes that are found in fewer than 3 cells
    adata.var['mt'] = adata.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    upper_lim = np.quantile(adata.obs.n_genes_by_counts.values, .98)
    lower_lim = np.quantile(adata.obs.n_genes_by_counts.values, .02)
    adata = adata[(adata.obs.n_genes_by_counts < upper_lim) & (adata.obs.n_genes_by_counts > lower_lim)]
    adata = adata[adata.obs.pct_counts_mt < 20]
    sc.pp.normalize_total(adata, target_sum=1e4) #normalize every cell to 10,000 UMI
    sc.pp.log1p(adata) #change to log counts
    sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5) #these are default values
    adata.raw = adata #save raw data before processing values and further filtering
    adata = adata[:, adata.var.highly_variable] #filter highly variable
    sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt']) #Regress out effects of total counts per cell and the percentage of mitochondrial genes expressed
    sc.pp.scale(adata, max_value=10) #scale each gene to unit variance
    sc.tl.pca(adata, svd_solver='arpack')
    sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20)
    sc.tl.leiden(adata, resolution = 0.25)
    sc.tl.umap(adata)
    sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
    
    
    #find markers
    results = adata.uns['rank_genes_groups']
    out = np.array([[0,0,0,0,0]])
    for group in results['names'].dtype.names:
        out = np.vstack((out, np.vstack((results['names'][group],
                                         results['scores'][group],
                                         results['pvals_adj'][group],
                                         results['logfoldchanges'][group],
                                         np.array([group] * len(results['names'][group])).astype('object'))).T))
    markers = pd.DataFrame(out[1:], columns = ['Gene', 'scores', 'pval_adj', 'lfc', 'cluster'])
    adata.uns['markers'] = markers #save marker df to uns

    
    return adata

In [ ]:
# c elegans ncbi taxonomy id is 6239
# paste this into search at https://www.ncbi.nlm.nih.gov/gene 
# "6239"[Taxonomy ID] AND alive[property] AND genetype protein coding[Properties]
# send to > file > text file > create file
# get gene_result.txt
# this is the background gene set

In [ ]:
# find this in environment from goatools
# /home/carl/miniconda3/envs/ontology/bin/ncbi_gene_results_to_python.py

In [1]:
# -o output
!python /home/carl/miniconda3/envs/ontology/bin/ncbi_gene_results_to_python.py -o genes_ncbi_c_elegans_proteincoding.py gene_result.txt

      19,983 lines READ:  gene_result.txt
      19,983 geneids WROTE: genes_ncbi_c_elegans_proteincoding.py


In [2]:
# import from genes_ncbi_c_elegans_proteincoding.py
# can move file to python path
from genes_ncbi_c_elegans_proteincoding import GENEID2NT as GeneID2nt_elegans

In [3]:
from goatools.base import download_go_basic_obo
from goatools.base import download_ncbi_associations
from goatools.obo_parser import GODag
from goatools.anno.genetogo_reader import Gene2GoReader
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS

In [4]:
obo_fname = download_go_basic_obo()
fin_gene2go = download_ncbi_associations()
obodag = GODag("go-basic.obo")

  EXISTS: go-basic.obo
  EXISTS: gene2go
go-basic.obo: fmt(1.2) rel(2026-07-26) 41,378 Terms


In [5]:
# need to unzip gene2go.gz 
# not sure whats different here than in guide
# this file is massive
# don't rerun if possible
!gunzip gene2go.gz -kf

In [6]:
# test find daf-16 gene id: 172981
GeneID2nt_elegans[172981]

ntncbi(tax_id=6239, Org_name='Caenorhabditis elegans', GeneID=172981, CurrentID=0, Status='live', Symbol='daf-16', Aliases=['CELE_R13H8.1'], description='Forkhead box protein O', other_designations='Forkhead box protein O', map_location='', chromosome='I', genomic_nucleotide_accession_version='NC_003279.8', start_position_on_the_genomic_accession=10750498, end_position_on_the_genomic_accession=10776703, orientation='plus', exon_count=15, OMIM=[], no_hdr0='')

In [5]:
# new dictionary
# gene names and their id
mapper = {}

for key in GeneID2nt_elegans:
    mapper[GeneID2nt_elegans[key].Symbol] = GeneID2nt_elegans[key].GeneID

# mapper but swap keys and values
inv_map = {v: k for k, v in mapper.items()}

In [6]:
mapper

{'homt-1': 171590,
 'nlp-40': 171591,
 'rcor-1': 171592,
 'sesn-1': 171593,
 'pgs-1': 171594,
 'Y48G1C.5': 171595,
 'Y48G1C.6': 171597,
 'pid-2': 171599,
 'rab-11.1': 171601,
 'rpl-7': 171602,
 'F53G12.9': 171603,
 'F53G12.8': 171604,
 'col-45': 171605,
 'spe-8': 171606,
 'mex-3': 171607,
 'bli-3': 171608,
 'ptr-11': 171609,
 'cest-27': 171610,
 'F56C11.3': 171611,
 'snpc-3.2': 171615,
 'marc-4': 171616,
 'ztf-3': 171617,
 'C53D5.1': 171618,
 'C53D5.5': 171619,
 'xpo-2': 171621,
 'nol-14': 171622,
 'daf-25': 171623,
 'mbtr-1': 171624,
 'Y48G1A.2': 171625,
 'rnp-8': 171627,
 'R119.3': 171628,
 'taf-4': 171630,
 'R119.5': 171631,
 'otub-2': 171633,
 'C07F11.2': 171634,
 'tol-1': 171635,
 'W04C9.4': 171636,
 'cutl-13': 171637,
 'W04C9.2': 171638,
 'haf-4': 171639,
 'Y65B4BL.3': 171640,
 'Y65B4BL.4': 171641,
 'deps-1': 171642,
 'acs-13': 171643,
 'Y65B4BL.1': 171644,
 'psf-3': 171645,
 'icd-2': 171646,
 'wwp-1': 171647,
 'lpr-1': 171648,
 'grl-16': 171649,
 'hum-7': 171650,
 'eme-1': 17165

In [7]:
# make sure gene2go isn't empty
# gene2go is a large file, don't rerun if possible
objanno = Gene2GoReader(fin_gene2go, taxids=[6239])
ns2assoc = objanno.get_ns2assc()

HMS:0:01:27.185341  92,559 annotations, 11,791 genes,  6,767 GOs, 1 taxids READ: gene2go 


In [8]:
# dictionary of dictionaries
# MF, CC, BP for the three GO aspects
# shows gene id and thier go associations
ns2assoc

{'CC': {171590: {'GO:0005737'},
  171591: {'GO:0005576', 'GO:0031410'},
  171592: {'GO:0000118', 'GO:0005634', 'GO:0005667'},
  171593: {'GO:0005634', 'GO:0005737'},
  171594: {'GO:0005737', 'GO:0005739'},
  171597: {'GO:0005634'},
  171599: {'GO:0048471'},
  171601: {'GO:0000139',
   'GO:0005768',
   'GO:0005794',
   'GO:0005813',
   'GO:0005819',
   'GO:0005829',
   'GO:0016324',
   'GO:0030133',
   'GO:0055037',
   'GO:0055038',
   'GO:0060473',
   'GO:1990676'},
  171602: {'GO:0022625'},
  171606: {'GO:0005737', 'GO:0005886'},
  171607: {'GO:0005634', 'GO:0005737', 'GO:0043186'},
  171608: {'GO:0005886', 'GO:0016324', 'GO:0043020', 'GO:1990204'},
  171609: {'GO:0005886', 'GO:0016020', 'GO:0030659'},
  171611: {'GO:0005739', 'GO:0005758'},
  171615: {'GO:0005634', 'GO:0019185'},
  171616: {'GO:0005737', 'GO:0005764', 'GO:0031901', 'GO:0031902'},
  171617: {'GO:0005634', 'GO:0005654'},
  171619: {'GO:0005886'},
  171621: {'GO:0005634', 'GO:0005635', 'GO:0005737', 'GO:0005829'},
  171

In [18]:
goeaobj = GOEnrichmentStudyNS(
    GeneID2nt_elegans.keys(),
    ns2assoc,
    obodag,
    propagate_counts = False,
    alpha = 0.05,
    methods = ['fdr_bh']
)


Load BP Ontology Enrichment Analysis ...
 46%  9,279 of 19,983 population items found in association

Load CC Ontology Enrichment Analysis ...
 50%  9,901 of 19,983 population items found in association

Load MF Ontology Enrichment Analysis ...
 48%  9,529 of 19,983 population items found in association


In [19]:
GO_items = []

temp = goeaobj.ns2objgoea['BP'].assoc
for item in temp:
    GO_items += temp[item]

temp = goeaobj.ns2objgoea['CC'].assoc
for item in temp:
    GO_items += temp[item]

temp = goeaobj.ns2objgoea['MF'].assoc
for item in temp:
    GO_items += temp[item]

In [20]:
GO_items.count('GO:0000139')

179

In [21]:
# looks like you don't need any of the DGE stats, just a list of genes
# need gene names from smallgenes
# try ce.1.1 WBGene00009091

In [22]:
GeneID2nt_elegans[172981]

ntncbi(tax_id=6239, Org_name='Caenorhabditis elegans', GeneID=172981, CurrentID=0, Status='live', Symbol='daf-16', Aliases=['CELE_R13H8.1'], description='Forkhead box protein O', other_designations='Forkhead box protein O', map_location='', chromosome='I', genomic_nucleotide_accession_version='NC_003279.8', start_position_on_the_genomic_accession=10750498, end_position_on_the_genomic_accession=10776703, orientation='plus', exon_count=15, OMIM=[], no_hdr0='')

In [23]:
#mapper['Y73B2A.8']

In [24]:
#mapper

In [25]:
import os

# Get the current working directory
cwd = os.getcwd()
print(cwd)

/home/carl/Code/isoforms/APCanalysis/go


In [26]:
import pandas as pd
sg_syms = pd.read_csv('sg_symbols.csv', header=None)

In [27]:
sg_syms = sg_syms.rename(columns={0: 'wbgene', 1: 'symbol'})
sg_syms

,wbgene,symbol
0,WBGene00007958,C35C5.9
1,WBGene00009964,fip-6
2,WBGene00194788,R02D5.10
3,WBGene00004810,skr-4
4,WBGene00002107,ins-24
...,...,...
916,WBGene00008860,romo-1
917,WBGene00009714,F44G4.5
918,WBGene00001235,elb-1
919,WBGene00008807,F14F3.4


In [34]:
#pass list of gene symbols
def go_it(test_genes):
    print(f'input genes: {len(test_genes)}')
    
    mapped_genes = []
    for gene in test_genes:
        try:
            mapped_genes.append(mapper[gene])
        except:
            pass
    print(f'mapped genes: {len(mapped_genes)}')
    
    goea_results_all = goeaobj.run_study(mapped_genes)
    goea_results_sig = [r for r in goea_results_all if r.p_fdr_bh < 0.05]
    GO = pd.DataFrame(list(map(lambda x: [x.GO, x.goterm.name, x.goterm.namespace, x.p_uncorrected, x.p_fdr_bh,\
                   x.ratio_in_study[0], x.ratio_in_study[1], GO_items.count(x.GO), list(map(lambda y: inv_map[y], x.study_items)),\
                   ], goea_results_sig)), columns = ['GO', 'term', 'class', 'p', 'p_corr', 'n_genes',\
                                                    'n_study', 'n_go', 'study_genes'])

    GO = GO[GO.n_genes > 1]
    return GO

In [29]:
#from statsmodels.sandbox.stats.multicomp import multipletests
from statsmodels.stats.multitest import multipletests

In [35]:
resdf = go_it(sg_syms.symbol.values)

input genes: 921
mapped genes: 852

Runing BP Ontology Analysis: current study set of 852 IDs.
 31%    262 of    852 study items found in association
100%    852 of    852 study items found in population(19983)
Calculating 3,522 uncorrected p-values using fisher_scipy_stats
   3,522 terms are associated with  9,279 of 19,983 population items
     342 terms are associated with    262 of    852 study items
  METHOD fdr_bh:
       7 GO terms found significant (< 0.05=alpha) (  2 enriched +   5 purified): statsmodels fdr_bh
      45 study items associated with significant GO IDs (enriched)
      19 study items associated with significant GO IDs (purified)

Runing CC Ontology Analysis: current study set of 852 IDs.
 32%    272 of    852 study items found in association
100%    852 of    852 study items found in population(19983)
Calculating 963 uncorrected p-values using fisher_scipy_stats
     963 terms are associated with  9,901 of 19,983 population items
     183 terms are associated wit

In [31]:
mapped_genes = []
test_genes = sg_syms.symbol.values
for gene in test_genes:
    try:
        mapped_genes.append(mapper[gene])
    except:
        pass

mapped_genes

[183231,
 172707,
 13216575,
 191762,
 173266,
 191238,
 34700616,
 180711,
 184487,
 187236,
 3565538,
 177731,
 187890,
 186714,
 178637,
 266832,
 172380,
 183436,
 3565652,
 172002,
 13215068,
 172696,
 3896870,
 3896848,
 179510,
 183004,
 174660,
 175566,
 180334,
 177168,
 191634,
 181222,
 173941,
 175793,
 180877,
 185804,
 183827,
 184441,
 183138,
 24105173,
 174884,
 184172,
 4927032,
 13185074,
 13206052,
 13182797,
 3565667,
 181916,
 173805,
 178648,
 186933,
 179476,
 180477,
 173507,
 187788,
 178333,
 178712,
 186846,
 3896864,
 6418745,
 179060,
 62532686,
 13183371,
 176062,
 185330,
 180777,
 171763,
 184896,
 181548,
 173765,
 184616,
 186221,
 188821,
 177472,
 173492,
 174700,
 190280,
 191132,
 190395,
 4363108,
 176100,
 28661654,
 3565685,
 177231,
 6418652,
 3565227,
 180025,
 182612,
 191537,
 176452,
 175273,
 3565918,
 173517,
 185265,
 27219624,
 3565869,
 176610,
 179599,
 3565721,
 190768,
 177488,
 176290,
 13186216,
 177550,
 3565090,
 185346,
 35650

In [32]:
# not sure how to fix error, try to follow another guide
# error fixed thanks to copilot 
# had to manually edit import at:
# lib/python3.14/site-packages/goatools/multiple_testing.py
# change this
# from statsmodels.sandbox.stats.multicomp import multipletests
# to this
# from statsmodels.stats.multitest import multipletests

In [33]:
# does the same the same thing as go_it from Sanbomics
# but go_it probably edits some parameters
res = goeaobj.run_study(mapped_genes)
res


Runing BP Ontology Analysis: current study set of 852 IDs.
 31%    262 of    852 study items found in association
100%    852 of    852 study items found in population(19983)
Calculating 3,522 uncorrected p-values using fisher_scipy_stats
   3,522 terms are associated with  9,279 of 19,983 population items
     342 terms are associated with    262 of    852 study items
  METHOD fdr_bh:
       7 GO terms found significant (< 0.05=alpha) (  2 enriched +   5 purified): statsmodels fdr_bh
      45 study items associated with significant GO IDs (enriched)
      19 study items associated with significant GO IDs (purified)

Runing CC Ontology Analysis: current study set of 852 IDs.
 32%    272 of    852 study items found in association
100%    852 of    852 study items found in population(19983)
Calculating 963 uncorrected p-values using fisher_scipy_stats
     963 terms are associated with  9,901 of 19,983 population items
     183 terms are associated with    272 of    852 study items
  ME

[GOEnrichmentRecord(GO:0006412),
 GOEnrichmentRecord(GO:0050829),
 GOEnrichmentRecord(GO:0045039),
 GOEnrichmentRecord(GO:0002181),
 GOEnrichmentRecord(GO:0045041),
 GOEnrichmentRecord(GO:0006363),
 GOEnrichmentRecord(GO:0006362),
 GOEnrichmentRecord(GO:0050832),
 GOEnrichmentRecord(GO:0006361),
 GOEnrichmentRecord(GO:0042790),
 GOEnrichmentRecord(GO:1990115),
 GOEnrichmentRecord(GO:1990113),
 GOEnrichmentRecord(GO:1990114),
 GOEnrichmentRecord(GO:0035915),
 GOEnrichmentRecord(GO:0030490),
 GOEnrichmentRecord(GO:0051604),
 GOEnrichmentRecord(GO:0042026),
 GOEnrichmentRecord(GO:1902533),
 GOEnrichmentRecord(GO:1905910),
 GOEnrichmentRecord(GO:0051382),
 GOEnrichmentRecord(GO:0006122),
 GOEnrichmentRecord(GO:0006366),
 GOEnrichmentRecord(GO:0019941),
 GOEnrichmentRecord(GO:0006879),
 GOEnrichmentRecord(GO:0000387),
 GOEnrichmentRecord(GO:0007080),
 GOEnrichmentRecord(GO:0009408),
 GOEnrichmentRecord(GO:0032469),
 GOEnrichmentRecord(GO:0051984),
 GOEnrichmentRecord(GO:0015908),
 GOEnrichm

In [64]:
sg_syms.symbol.values

<StringArray>
[  'C35C5.9',   'F53B6.9',  'R02D5.10', 'Y60A3A.18',   'ZC334.3',   'ZK177.9',
   'C54F6.6',  'F14H12.6',  'F14F11.2',   'K09G1.2',
 ...
  'R07E5.19',  'C33A12.4', 'F26D10.12',   'F55H2.8',   'C45B2.8',   'F15D4.3',
   'F44G4.5', 'Y41C4A.10',   'F14F3.4',   'R07E3.2']
Length: 921, dtype: str

In [37]:
not_in = 0
for sym in sg_syms.symbol.values:
    if sym in mapper:
        #print(sym)
        continue
    else:
        #print(sym, 'not in')
        not_in += 1
not_in

69

In [55]:
mapper

{'homt-1': 171590,
 'nlp-40': 171591,
 'rcor-1': 171592,
 'sesn-1': 171593,
 'pgs-1': 171594,
 'Y48G1C.5': 171595,
 'Y48G1C.6': 171597,
 'pid-2': 171599,
 'rab-11.1': 171601,
 'rpl-7': 171602,
 'F53G12.9': 171603,
 'F53G12.8': 171604,
 'col-45': 171605,
 'spe-8': 171606,
 'mex-3': 171607,
 'bli-3': 171608,
 'ptr-11': 171609,
 'cest-27': 171610,
 'F56C11.3': 171611,
 'snpc-3.2': 171615,
 'marc-4': 171616,
 'ztf-3': 171617,
 'C53D5.1': 171618,
 'C53D5.5': 171619,
 'xpo-2': 171621,
 'nol-14': 171622,
 'daf-25': 171623,
 'mbtr-1': 171624,
 'Y48G1A.2': 171625,
 'rnp-8': 171627,
 'R119.3': 171628,
 'taf-4': 171630,
 'R119.5': 171631,
 'otub-2': 171633,
 'C07F11.2': 171634,
 'tol-1': 171635,
 'W04C9.4': 171636,
 'cutl-13': 171637,
 'W04C9.2': 171638,
 'haf-4': 171639,
 'Y65B4BL.3': 171640,
 'Y65B4BL.4': 171641,
 'deps-1': 171642,
 'acs-13': 171643,
 'Y65B4BL.1': 171644,
 'psf-3': 171645,
 'icd-2': 171646,
 'wwp-1': 171647,
 'lpr-1': 171648,
 'grl-16': 171649,
 'hum-7': 171650,
 'eme-1': 17165